# Autoresearch Experiment Analysis

Analysis of autonomous RL hyperparameter tuning results from `results.tsv`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the TSV (tab-separated, 5 columns: commit, win_rate, route_pct, status, description)
df = pd.read_csv("results.tsv", sep="\t")
df["win_rate"]  = pd.to_numeric(df["win_rate"],  errors="coerce")
df["route_pct"] = pd.to_numeric(df["route_pct"], errors="coerce")
df["status"]    = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep    = counts.get("KEEP",    0)
n_discard = counts.get("DISCARD", 0)
n_crash   = counts.get("CRASH",   0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

In [ ]:
# Show all KEPT experiments
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    print(f"  #{i:3d}  win_rate={row['win_rate']:.4f}  route_pct={row['route_pct']:.1f}%  {row['description']}")

## Win Rate Over Time

Track how the best (kept) win rate evolves as experiments progress. The running maximum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = df[df["status"] != "CRASH"].copy().reset_index(drop=True)

baseline_win = valid.loc[0, "win_rate"]

# Plot discarded as faint background dots
disc = valid[valid["status"] == "DISCARD"]
ax.scatter(disc.index, disc["win_rate"],
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Plot kept experiments as prominent green dots
kept_v = valid[valid["status"] == "KEEP"]
ax.scatter(kept_v.index, kept_v["win_rate"],
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line (higher is better)
kept_mask = valid["status"] == "KEEP"
kept_idx  = valid.index[kept_mask]
kept_win  = valid.loc[kept_mask, "win_rate"]
running_max = kept_win.cummax()
ax.step(kept_idx, running_max, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for idx, win in zip(kept_idx, kept_win):
    desc = str(valid.loc[idx, "description"]).strip()
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (idx, win),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

# Baseline reference line
ax.axhline(baseline_win, color="#aaaaaa", linewidth=1, linestyle="--", label=f"Baseline ({baseline_win:.3f})")

n_total = len(df)
n_kept  = len(df[df["status"] == "KEEP"])
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Win Rate vs example_agent (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = df[df["status"] == "KEEP"].copy()
baseline_win  = df.iloc[0]["win_rate"]
baseline_route = df.iloc[0]["route_pct"]
best_win  = kept["win_rate"].max()
best_row  = kept.loc[kept["win_rate"].idxmax()]

print(f"Baseline win_rate:   {baseline_win:.4f}  ({baseline_win:.1%})")
print(f"Best win_rate:       {best_win:.4f}  ({best_win:.1%})")
print(f"Total improvement:   {best_win - baseline_win:+.4f}  ({(best_win - baseline_win) * 100:+.1f} pp)")
print(f"Best experiment:     {best_row['description']}")
print()

print("Cumulative effort per improvement:")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    print(f"  Experiment #{row['index']:3d}: win_rate={row['win_rate']:.4f}  route_pct={row['route_pct']:.1f}%  {row['description']}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta vs the previous kept experiment
kept = df[df["status"] == "KEEP"].copy()
kept["prev_win"] = kept["win_rate"].shift(1)
kept["delta"]    = kept["win_rate"] - kept["prev_win"]

# Drop baseline (no delta)
hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta':>8}  {'Win Rate':>9}  {'Route%':>7}  Description")
print("-" * 80)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.4f}  {row['win_rate']:.4f}     {row['route_pct']:>5.1f}%  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.4f}  {'':>9}  {'':>7}  TOTAL improvement over baseline")

## Win Rate vs Route Completion

Scatter plot of all non-crashed experiments. Good experiments should be in the top-right.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

valid = df[df["status"] != "CRASH"].copy()

colors = {"KEEP": "#2ecc71", "DISCARD": "#cccccc"}
for status, grp in valid.groupby("status"):
    ax.scatter(grp["route_pct"], grp["win_rate"],
               c=colors.get(status, "#aaaaaa"),
               s=40, alpha=0.7, label=status.capitalize(),
               edgecolors="black", linewidths=0.4)

# Label kept experiments
for _, row in valid[valid["status"] == "KEEP"].iterrows():
    desc = str(row["description"])[:30]
    ax.annotate(desc, (row["route_pct"], row["win_rate"]),
                textcoords="offset points", xytext=(5, 4),
                fontsize=7.5, color="#1a7a3a")

ax.set_xlabel("Avg Route Completion %", fontsize=12)
ax.set_ylabel("Win Rate vs example_agent", fontsize=12)
ax.set_title("Win Rate vs Route Completion", fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)
ax.set_xlim(0, 105)
ax.set_ylim(-0.02, 1.02)

plt.tight_layout()
plt.savefig("scatter.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to scatter.png")